## Transformation

In [1]:
from google.cloud import bigquery
import pandas as pd

project_id = "linear-theater-436300-r9"
dataset_id = "ecommerce_pipeline"

client = bigquery.Client()

# Load product metadata
items_query = f"""
SELECT * FROM `{project_id}.{dataset_id}.cleaned_items`
"""
items_df = client.query(items_query).to_dataframe()

# Load C4 queries
queries_query = f"""
SELECT * FROM `{project_id}.{dataset_id}.raw_queries_c4`
"""
queries_df = client.query(queries_query).to_dataframe()

# Load sentiment reviews
sentiment_query = f"""
SELECT * FROM `{project_id}.{dataset_id}.raw_reviews_sentiment`
"""
sentiment_df = client.query(sentiment_query).to_dataframe()

items_df.shape, queries_df.shape, sentiment_df.shape


/home/niranjanrao07/cod-multiagent-ecommerce/.venv/lib/python3.12/site-packages/google/cloud/bigquery/table.py:1994: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


((1058417, 6), (21223, 6), (50000, 3))

## Clean Product Metadata

In [2]:
# Drop rows with missing item_id
items_df = items_df.dropna(subset=["item_id"])

# Strip whitespace from text fields
text_cols = ["title", "description", "brand", "category"]
for col in text_cols:
    items_df[col] = items_df[col].astype(str).str.strip()

# Drop items with extremely short titles (noise)
items_df = items_df[items_df["title"].str.len() > 3]

# Normalize category capitalization
items_df["category"] = items_df["category"].str.title()

# Remove duplicate item_ids (keep first)
items_df = items_df.drop_duplicates(subset=["item_id"])

# Reset index
items_df = items_df.reset_index(drop=True)

print("Cleaned items_df shape:", items_df.shape)
items_df.head()


Cleaned items_df shape: (1048018, 6)


,item_id,category,metadata,title,description,brand
0,B016SEPJXY,Kindle,"June Peters, You Will Change The World One Day...","June Peters, You Will Change The World One Day","About the Author Alika was born in Richmond, C...",June Peters
1,B00IQOFTR6,Kindle,The Silent Sister: A Novel. Review “ The Silen...,The Silent Sister: A Novel,Review “ The Silent Sister is a powerful and t...,The Silent Sister: A Novel
2,B00242VQMS,Office,"Old Time Couple Dances, Fiddle and Accordion. ...","Old Time Couple Dances, Fiddle and Accordion","Old Time Couple Dances, Fiddle and Accordion f...",Old Time Couple Dances
3,B0B4T46DBD,Office,"Five Star 2 Pocket Folders, 4 Pack, Plastic Fo...","Five Star 2 Pocket Folders, 4 Pack, Plastic Fo...",LASTS ALL YEAR. GUARANTEED!* This Five Star fo...,Five Star
4,B08JLRJPD7,Pet,Zozostore Dog Toothbrush Squeaky Chew Toys – 2...,Zozostore Dog Toothbrush Squeaky Chew Toys – 2...,Specifications Material:Natural RubberColor:Bl...,Zozostore Dog Toothbrush Squeaky Chew Toys –


## Clean Product Metadata

In [3]:
# Drop rows missing essential fields
queries_df = queries_df.dropna(subset=["query", "item_id"])

# Strip whitespace
queries_df["query"] = queries_df["query"].astype(str).str.strip()
queries_df["ori_review"] = queries_df["ori_review"].astype(str).str.strip()

# Remove empty/short queries (noise)
queries_df = queries_df[queries_df["query"].str.len() > 10]

# Add simple features
queries_df["query_length"] = queries_df["query"].str.len()
queries_df["query_word_count"] = queries_df["query"].str.split().str.len()

# Reset index
queries_df = queries_df.reset_index(drop=True)

print("Cleaned queries_df shape:", queries_df.shape)
queries_df.head()


Cleaned queries_df shape: (21223, 8)


,qid,query,item_id,user_id,ori_rating,ori_review,query_length,query_word_count
0,18857,I'm looking for a modern makeover story that's...,0007582471,AGLPRIH7NGIMA7ZR2A4SBBFHVKWQ,5,Witty and fun modern makeover story. There's a...,428,79
1,387,I need to find a unique history book that is n...,0060735880,AE5JQ5EWG6SCUQTTASPSDVF5ND7A,5,Out of Print This is a terrific unique history...,289,60
2,1036,I'm looking for a book with beautiful words an...,0061730793,AFPZ7WCASBA67MZAYNJYJN5MHTBA,5,Beautiful! This book has been wonderfully done...,146,27
3,15451,I am looking for a book for my two-year-old wh...,0061900621,AGVS2PQTKJZ2IFGVJVMMLCPL7UWQ,5,Immediate hit with 2 yr old My newly minted tw...,214,41
4,16882,"I'm looking for a great book, but I want to av...",0062124277,AGXN5ZI5CYIAX34Z6DJBSGP777LA,5,"Inspired as always Great book, of course, but ...",156,33


## Clean Sentiment Dataset

In [4]:
# Drop missing content
sentiment_df = sentiment_df.dropna(subset=["content"])

# Strip whitespace from text fields
sentiment_df["title"] = sentiment_df["title"].astype(str).str.strip()
sentiment_df["content"] = sentiment_df["content"].astype(str).str.strip()

# Normalize label (already 0 or 1)
sentiment_df["label"] = sentiment_df["label"].astype(int)

# Remove empty or very short content
sentiment_df = sentiment_df[sentiment_df["content"].str.len() > 10]

# Add text length features
sentiment_df["content_length"] = sentiment_df["content"].str.len()
sentiment_df["word_count"] = sentiment_df["content"].str.split().str.len()

# Reset index
sentiment_df = sentiment_df.reset_index(drop=True)

print("Cleaned sentiment_df shape:", sentiment_df.shape)
sentiment_df.head()


Cleaned sentiment_df shape: (50000, 5)


,label,title,content,content_length,word_count
0,0,Buyer beware,"This is a self-published book, and if you want...",724,137
1,0,The Worst!,A complete waste of time. Typographical errors...,204,33
2,0,Oh please,I guess you have to be a romance novel lover f...,481,91
3,0,Awful beyond belief!,I feel I have to write to keep others from was...,670,129
4,0,Don't try to fool us with fake reviews.,It's glaringly obvious that all of the glowing...,263,45


## Feature Engineering for Product Metadata

In [5]:
import re

# Text length features
items_df["title_length"] = items_df["title"].str.len()
items_df["description_length"] = items_df["description"].str.len()
items_df["metadata_length"] = items_df["metadata"].str.len()

# Simple price extraction heuristic
# Looks for patterns like "$12.99", "12.99", "19", "19.99" in metadata text.
def extract_price(text):
    if not isinstance(text, str):
        return None
    # Look for $XX.XX or XX.XX
    match = re.search(r"\$?(\d+\.\d{1,2})", text)
    if match:
        return float(match.group(1))
    # Look for integer prices like $19 or 19
    match = re.search(r"\$?(\d{1,4})\b", text)
    if match:
        value = float(match.group(1))
        # avoid nonsense matches (e.g., years like 2020)
        if 1 <= value <= 2000:
            return value
    return None

items_df["price"] = items_df["metadata"].apply(extract_price)

# Price segment (bucket)
def price_segment(x):
    if x is None:
        return "unknown"
    if x < 10:
        return "low"
    if x < 30:
        return "medium"
    if x < 100:
        return "high"
    return "premium"

items_df["price_segment"] = items_df["price"].apply(price_segment)

# Summary
print(items_df[["item_id", "title", "price", "price_segment"]].head())
print("Metadata with price extracted:", items_df["price"].notna().sum())


      item_id                                              title   price  \
0  B016SEPJXY     June Peters, You Will Change The World One Day    32.0   
1  B00IQOFTR6                         The Silent Sister: A Novel     NaN   
2  B00242VQMS       Old Time Couple Dances, Fiddle and Accordion  1961.0   
3  B0B4T46DBD  Five Star 2 Pocket Folders, 4 Pack, Plastic Fo...     2.0   
4  B08JLRJPD7  Zozostore Dog Toothbrush Squeaky Chew Toys – 2...     NaN   

  price_segment  
0          high  
1       premium  
2       premium  
3           low  
4       premium  
Metadata with price extracted: 729451


In [6]:
import re

# Additional text cleaning
def clean_query(text):
    if not isinstance(text, str):
        return ""
    text = text.strip()
    text = re.sub(r"\s+", " ", text)  # collapse multiple spaces
    text = text.replace("\n", " ")
    return text

queries_df["query_clean"] = queries_df["query"].apply(clean_query)

# Feature: average word length
queries_df["avg_word_length"] = (
    queries_df["query_clean"].str.replace(r"[^A-Za-z0-9 ]", "", regex=True)
                              .str.split()
                              .apply(lambda x: sum(len(w) for w in x) / len(x) if len(x) > 0 else 0)
)

# Feature: query type (simple heuristic)
def classify_query(text):
    text_lower = text.lower()
    if any(x in text_lower for x in ["look for", "need", "want", "find"]):
        return "intent_search"
    if "recommend" in text_lower:
        return "recommendation_request"
    return "other"

queries_df["query_type"] = queries_df["query_clean"].apply(classify_query)

# Verify
print(queries_df[["qid", "query_clean", "query_type", "avg_word_length"]].head())
print("Final queries_df shape:", queries_df.shape)


     qid                                        query_clean     query_type  \
0  18857  I'm looking for a modern makeover story that's...  intent_search   
1    387  I need to find a unique history book that is n...  intent_search   
2   1036  I'm looking for a book with beautiful words an...  intent_search   
3  15451  I am looking for a book for my two-year-old wh...  intent_search   
4  16882  I'm looking for a great book, but I want to av...  intent_search   

   avg_word_length  
0         4.291139  
1         3.733333  
2         4.296296  
3         4.097561  
4         3.606061  
Final queries_df shape: (21223, 11)


## Feature Engineering for Sentiment Dataset

In [7]:
import re

# Clean the content text
def clean_text(text):
    if not isinstance(text, str):
        return ""
    text = text.strip()
    text = re.sub(r"\s+", " ", text)        # collapse spaces
    text = text.replace("\n", " ")          # remove newlines
    return text

sentiment_df["content_clean"] = sentiment_df["content"].apply(clean_text)

# Remove non-alphanumeric characters except punctuation
sentiment_df["content_clean"] = sentiment_df["content_clean"].str.replace(
    r"[^A-Za-z0-9.,!?;:()'\"\s]", "", regex=True
)

# Heuristic: sentiment strength
def sentiment_strength(label, length):
    if label == 1:
        return min(1.0, length / 500)   # stronger positive for long reviews
    else:
        return min(1.0, length / 300)   # negative reviews tend to be shorter

sentiment_df["sentiment_strength"] = sentiment_df.apply(
    lambda row: sentiment_strength(row["label"], row["content_length"]),
    axis=1
)

# Verify
print(sentiment_df[["label", "content_clean", "sentiment_strength"]].head())
print("Final sentiment_df shape:", sentiment_df.shape)


   label                                      content_clean  \
0      0  This is a selfpublished book, and if you want ...   
1      0  A complete waste of time. Typographical errors...   
2      0  I guess you have to be a romance novel lover f...   
3      0  I feel I have to write to keep others from was...   
4      0  It's glaringly obvious that all of the glowing...   

   sentiment_strength  
0            1.000000  
1            0.680000  
2            1.000000  
3            1.000000  
4            0.876667  
Final sentiment_df shape: (50000, 7)


## Upload Cleaned Product Metadata to BigQuery

In [8]:
from google.cloud import bigquery

project_id = "linear-theater-436300-r9"
dataset_id = "ecommerce_pipeline"

client = bigquery.Client()

table_id_items = f"{project_id}.{dataset_id}.products_cleaned"

job_config = bigquery.LoadJobConfig(
    write_disposition=bigquery.WriteDisposition.WRITE_TRUNCATE,
    source_format=bigquery.SourceFormat.PARQUET
)

# Save parquet temp file
items_df.to_parquet("data/products_cleaned.parquet", index=False)

with open("data/products_cleaned.parquet", "rb") as f:
    job = client.load_table_from_file(f, table_id_items, job_config=job_config)

job.result()

table = client.get_table(table_id_items)
print("Loaded rows:", table.num_rows)
print("Columns:", [c.name for c in table.schema])


Loaded rows: 1048018
Columns: ['item_id', 'category', 'metadata', 'title', 'description', 'brand', 'title_length', 'description_length', 'metadata_length', 'price', 'price_segment']


## Upload Cleaned Queries to BigQuery

In [9]:
from google.cloud import bigquery

project_id = "linear-theater-436300-r9"
dataset_id = "ecommerce_pipeline"
table_full_id = f"{project_id}.{dataset_id}.queries_cleaned"

client = bigquery.Client()

# Save to Parquet
queries_df.to_parquet("data/queries_cleaned.parquet", index=False)

job_config = bigquery.LoadJobConfig(
    source_format=bigquery.SourceFormat.PARQUET,
    write_disposition=bigquery.WriteDisposition.WRITE_TRUNCATE
)

# Upload to BigQuery
with open("data/queries_cleaned.parquet", "rb") as f:
    load_job = client.load_table_from_file(f, table_full_id, job_config=job_config)

load_job.result()

# Verify upload
table = client.get_table(table_full_id)
print("Loaded rows:", table.num_rows)
print("Columns:", [c.name for c in table.schema])


Loaded rows: 21223
Columns: ['qid', 'query', 'item_id', 'user_id', 'ori_rating', 'ori_review', 'query_length', 'query_word_count', 'query_clean', 'avg_word_length', 'query_type']


## Upload cleaned sentiment dataset

In [ ]:
from google.cloud import bigquery

project_id = "linear-theater-436300-r9"
dataset_id = "ecommerce_pipeline"
table_full_id = f"{project_id}.{dataset_id}.reviews_cleaned"

client = bigquery.Client()

# Save cleaned sentiment dataset to Parquet
sentiment_df.to_parquet("data/reviews_cleaned.parquet", index=False)

job_config = bigquery.LoadJobConfig(
    source_format=bigquery.SourceFormat.PARQUET,
    write_disposition=bigquery.WriteDisposition.WRITE_TRUNCATE
)

# Upload to BigQuery
with open("data/reviews_cleaned.parquet", "rb") as f:
    load_job = client.load_table_from_file(
        f, table_full_id, job_config=job_config
    )

load_job.result()

# Verify upload
table = client.get_table(table_full_id)
print("Loaded rows:", table.num_rows)
print("Columns:", [c.name for c in table.schema])


Loaded rows: 50000
Columns: ['label', 'title', 'content', 'content_length', 'word_count', 'content_clean', 'sentiment_strength']


: 